# M3 · Reentrenar llantas con datasets de Roboflow Universe
**IAM-811 · Laboratorio 3**

El M3 original (5 keypoints por llanta: **centro, abajo, arriba, izquierda, derecha**) detecta bien llantas de autos,
pero casi no las de camiones y buses. Este notebook:

1. Descarga **tu dataset de llantas** + dos datasets públicos de Roboflow Universe con llantas de camión:
   - [axle](https://universe.roboflow.com/eternal-wiecw/axle-bjpov) (eternal): llantas de camión de cerca.
   - [truck axel](https://universe.roboflow.com/rajan-28noy/truck-axel) (rajan, CC BY 4.0): camiones en casetas de peaje.
2. Convierte cada caja/polígono de llanta a los **mismos 5 keypoints** (centro y puntos medios de los bordes).
3. Junta todo y hace *fine-tuning* desde tu `M3_llantas_best.pt`.
4. Mide el **error en el conteo de ejes** con los camiones de *truck axel* (traen el número real de ejes).

**Entorno de ejecución → GPU T4.** Secreto de Colab `ROBOFLOW_API_KEY` (icono de llave).

In [ ]:
!pip install -q ultralytics==8.3.240 roboflow==1.1.66 "pillow-heif<1.0"
import sys, torch, ultralytics
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda,
      '| gpu', torch.cuda.is_available() and torch.cuda.get_device_name(0), '| ultralytics', ultralytics.__version__)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, re, shutil
SALIDA = '/content/drive/MyDrive/peaje_evidencias'
os.makedirs(SALIDA, exist_ok=True)
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
# Tus proyectos (para copiar el nombre exacto del de llantas):
ws = rf.workspace()
print('workspace:', ws.url)
for p in ws.projects():
    print('  ', p)

In [ ]:
# Proyecto de llantas (keypoints): se detecta solo; si no, escríbelo a mano.
MI_WORKSPACE = ws.url
candidatos = [str(p).split('/')[-1] for p in ws.projects()
              if re.search(r'llant|rueda|wheel|tire|tyre|eje|axle', str(p), re.I)]
MI_PROYECTO = candidatos[0] if candidatos else 'CAMBIAR'   # p. ej. 'llantas-xxxx' (ver lista de arriba)
MI_VERSION = None                      # None = última versión; o el número que usaste para entrenar M3
print('proyecto de llantas:', MI_PROYECTO, '| candidatos:', candidatos)

# Pesos del M3 actual (inferencia/modelos/m3_pose.pt del repo).
# Si no están en Drive como peaje_evidencias/M3_llantas_best.pt, se piden aquí.
PESOS_BASE = f'{SALIDA}/M3_llantas_best.pt'
if not os.path.exists(PESOS_BASE):
    alt = f'{SALIDA}/m3_pose.pt'
    if os.path.exists(alt):
        PESOS_BASE = alt
    else:
        from google.colab import files
        print('Sube m3_pose.pt (carpeta inferencia/modelos del repo):')
        subido = files.upload()
        if subido:
            nombre = next(iter(subido))
            shutil.move(nombre, PESOS_BASE)
print('pesos base:', PESOS_BASE, os.path.exists(PESOS_BASE))

MAX_POR_UNIVERSE = 700                 # imágenes máximas de cada dataset público (tiempo de entrenamiento)


In [ ]:
import re

def ultima_version(workspace, proyecto):
    proj = rf.workspace(workspace).project(proyecto)
    nums = [int(re.findall(r'(\d+)$', str(v.version))[0]) for v in proj.versions()]
    return proj, max(nums)

if MI_VERSION is None:
    _, MI_VERSION = ultima_version(MI_WORKSPACE, MI_PROYECTO)
print('mi dataset:', MI_PROYECTO, 'versión', MI_VERSION)
mio = rf.workspace(MI_WORKSPACE).project(MI_PROYECTO).version(MI_VERSION).download('yolov8', location='/content/ds_mio')

descargas = {}
for nombre, (w, p) in {'axle': ('eternal-wiecw', 'axle-bjpov'), 'truckaxel': ('rajan-28noy', 'truck-axel')}.items():
    try:
        proj, v = ultima_version(w, p)
        descargas[nombre] = proj.version(v).download('yolov8', location=f'/content/ds_{nombre}')
        print(nombre, 'versión', v, 'OK')
    except Exception as e:
        print(nombre, 'NO se pudo descargar:', e)

In [ ]:
# Revisar las clases de cada dataset público para saber cuál es la llanta.
import yaml
for nombre, d in descargas.items():
    cfg = yaml.safe_load(open(f'{d.location}/data.yaml'))
    print(nombre, '->', cfg['names'])
print('mío ->', yaml.safe_load(open(f'{mio.location}/data.yaml')))

In [ ]:
# Clases que se consideran "llanta" en cada dataset público (ajusta si la celda anterior muestra otros nombres).
PATRON_LLANTA = re.compile(r'^(axles?|axels?|wheels?|tires?|tyres?|llantas?|ruedas?)$', re.I)
# En truck axel, las cajas del camión entero llevan el número de ejes: '2 axel', '3axel', ...
PATRON_CAMION = re.compile(r'^(\d)\s*-?\s*ax', re.I)

import glob, shutil, random
random.seed(42)
DEST = '/content/m3_mix'
for parte in ['train', 'valid']:
    os.makedirs(f'{DEST}/{parte}/images', exist_ok=True)
    os.makedirs(f'{DEST}/{parte}/labels', exist_ok=True)

def caja_desde_linea(partes):
    vals = list(map(float, partes[1:]))
    if len(vals) == 4:                       # caja YOLO: cx cy w h
        return vals
    xs, ys = vals[0::2], vals[1::2]          # polígono (segmentación): x1 y1 x2 y2 ...
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]

def a_pose(cx, cy, w, h):
    c = lambda v: min(max(v, 0.0), 1.0)
    pts = [(cx, cy), (cx, cy + h / 2), (cx, cy - h / 2), (cx - w / 2, cy), (cx + w / 2, cy)]  # centro, abajo, arriba, izq, der
    return f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f} ' + ' '.join(f'{c(x):.6f} {c(y):.6f} 2' for x, y in pts)

estadisticas = {}
ejes_reales = {}   # imagen -> lista de (caja_camion, n_ejes) para evaluar el conteo

for nombre, d in descargas.items():
    names = yaml.safe_load(open(f'{d.location}/data.yaml'))['names']
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    idx_llanta = {i for i, n in enumerate(names) if PATRON_LLANTA.match(str(n).strip())}
    idx_camion = {i: int(PATRON_CAMION.match(str(n).strip()).group(1)) for i, n in enumerate(names) if PATRON_CAMION.match(str(n).strip())}
    print(nombre, '| clases llanta:', [names[i] for i in idx_llanta], '| clases camión:', {names[i]: n for i, n in idx_camion.items()})
    etiquetas = []
    for parte in ['train', 'valid', 'test']:
        etiquetas += glob.glob(f'{d.location}/{parte}/labels/*.txt')
    random.shuffle(etiquetas)
    usadas = 0
    for lbl in etiquetas:
        lineas, camiones = [], []
        for l in open(lbl):
            partes = l.split()
            if len(partes) < 5:
                continue
            k = int(partes[0])
            if k in idx_llanta:
                lineas.append(a_pose(*caja_desde_linea(partes)))
            elif k in idx_camion:
                camiones.append((caja_desde_linea(partes), idx_camion[k]))
        if not lineas:
            continue
        base = os.path.splitext(os.path.basename(lbl))[0]
        img = next(iter(glob.glob(lbl.replace('/labels/', '/images/').rsplit('.', 1)[0] + '.*')), None)
        if not img:
            continue
        parte = 'valid' if random.random() < 0.15 else 'train'
        nuevo = f'{nombre}_{base}'
        shutil.copy(img, f'{DEST}/{parte}/images/{nuevo}{os.path.splitext(img)[1]}')
        open(f'{DEST}/{parte}/labels/{nuevo}.txt', 'w').write('\n'.join(lineas) + '\n')
        if camiones and parte == 'valid':
            ejes_reales[f'{DEST}/{parte}/images/{nuevo}{os.path.splitext(img)[1]}'] = camiones
        usadas += 1
        if usadas >= MAX_POR_UNIVERSE:
            break
    estadisticas[nombre] = usadas

# Tu dataset original (ya está en formato pose de 5 puntos): se copia tal cual.
for parte_origen, parte in [('train', 'train'), ('valid', 'valid'), ('test', 'valid')]:
    for img in glob.glob(f'{mio.location}/{parte_origen}/images/*'):
        base = os.path.splitext(os.path.basename(img))[0]
        lbl = f'{mio.location}/{parte_origen}/labels/{base}.txt'
        if os.path.exists(lbl):
            shutil.copy(img, f'{DEST}/{parte}/images/mio_{os.path.basename(img)}')
            shutil.copy(lbl, f'{DEST}/{parte}/labels/mio_{base}.txt')
estadisticas['mio'] = len(glob.glob(f'{DEST}/*/images/mio_*'))

cfg_mio = yaml.safe_load(open(f'{mio.location}/data.yaml'))
data = {
    'path': DEST, 'train': 'train/images', 'val': 'valid/images',
    'names': {0: 'llanta'}, 'kpt_shape': [5, 3],
    'flip_idx': cfg_mio.get('flip_idx', [0, 1, 2, 4, 3]),   # al voltear: izquierda <-> derecha
}
yaml.safe_dump(data, open(f'{DEST}/data.yaml', 'w'))
for parte in ['train', 'valid']:
    print(parte, len(glob.glob(f'{DEST}/{parte}/images/*')), 'imágenes')
print('por origen:', estadisticas, '| flip_idx:', data['flip_idx'], '| camiones con ejes reales en valid:', len(ejes_reales))

In [ ]:
# Revisión visual: los 5 puntos deben caer en el centro y los bordes de cada llanta.
import cv2, matplotlib.pyplot as plt
muestras = random.sample(glob.glob(f'{DEST}/train/images/*'), 6)
fig, ejes_plot = plt.subplots(2, 3, figsize=(15, 8))
for ax, img in zip(ejes_plot.flat, muestras):
    im = cv2.cvtColor(cv2.imread(img), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    base = os.path.splitext(os.path.basename(img))[0]
    for l in open(f'{DEST}/train/labels/{base}.txt'):
        v = list(map(float, l.split()))
        for j, color in enumerate([(255,0,0),(0,255,0),(0,0,255),(255,255,0),(255,0,255)]):
            cv2.circle(im, (int(v[5+3*j]*w), int(v[6+3*j]*h)), max(3, w//150), color, -1)
    ax.imshow(im); ax.set_title(base[:30]); ax.axis('off')
plt.show()

In [ ]:
from ultralytics import YOLO
modelo = YOLO(PESOS_BASE if os.path.exists(PESOS_BASE) else 'yolo11s-pose.pt')
modelo.train(data=f'{DEST}/data.yaml', epochs=40, imgsz=640, batch=16, patience=12,
             fliplr=0.5, project='/content/runs', name='m3_v2', seed=42, exist_ok=True)

In [ ]:
mejor = YOLO('/content/runs/m3_v2/weights/best.pt')
m = mejor.val(data=f'{DEST}/data.yaml', split='val')
print(f'POSE  mAP50={m.pose.map50:.3f}  mAP50-95={m.pose.map:.3f}')
print(f'CAJA  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}')

In [ ]:
# Error en el conteo de ejes (misma regla que el endpoint):
# cada llanta = un eje; centros a menos de media llanta (ruedas dobles) cuentan como uno.
import numpy as np

def contar(llantas):
    llantas.sort(key=lambda p: p[0]); ejes = []
    for p in llantas:
        if not ejes or p[0] - ejes[-1][0] > 0.5 * max(p[2], ejes[-1][2], 20):
            ejes.append(p)
    return len(ejes)

errores, exactos = [], 0
for img, camiones in ejes_reales.items():
    r = mejor.predict(img, conf=0.25, verbose=False)[0]
    h, w = r.orig_shape
    pred = [(kp[0][0], kp[0][1], b[2] - b[0]) for kp, b in zip(r.keypoints.xy.tolist(), r.boxes.xyxy.tolist())] if len(r.boxes) else []
    for (cx, cy, bw, bh), reales in camiones:
        x1, x2, y1, y2 = (cx - bw/2) * w, (cx + bw/2) * w, (cy - bh/2) * h, (cy + bh/2) * h
        dentro = [p for p in pred if x1 <= p[0] <= x2 and y1 <= p[1] <= y2]
        n = contar(dentro)
        errores.append(abs(n - reales)); exactos += int(n == reales)
if errores:
    print(f'Camiones evaluados: {len(errores)} | conteo exacto: {exactos/len(errores):.1%} | MAE ejes: {np.mean(errores):.2f}')
else:
    print('No hay camiones con número de ejes en validación (revisa PATRON_CAMION).')

In [ ]:
import shutil
shutil.copy('/content/runs/m3_v2/weights/best.pt', f'{SALIDA}/M3_llantas_v2.pt')
shutil.copy('/content/runs/m3_v2/weights/best.pt', '/content/m3_pose.pt')   # mismo nombre que usa la inferencia
shutil.copytree('/content/runs/m3_v2', f'{SALIDA}/m3_v2_run', dirs_exist_ok=True)
print('Guardado en Drive:', f'{SALIDA}/M3_llantas_v2.pt')
from google.colab import files
files.download('/content/m3_pose.pt')   # reemplaza inferencia/modelos/m3_pose.pt